In [25]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [26]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [27]:
simulator = BasicSimulator()

# Function to generate random bit using quantum circuit
def get_quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)

    # Extract the measured bit ('0' or '1') and convert to an integer
    measured_bit = list(counts.keys())[0]
    return int(measured_bit)

def generate_random_sequence(length):
    return [get_quantum_random_bit() for _ in range(length)]

In [28]:
# ==========================================
# ALICE
# ==========================================
def encode_message(bits, bases):
    encoded_qubits = []

    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)
        if bit == 1:
            qc.x(0) # Flips |0> to |1>
        if basis == 1:
            qc.h(0) # Changes basis (|0>, |1>) to (|+>, |->)

        encoded_qubits.append(qc)

    return encoded_qubits

# --- Alice Steps ---
# Define initial sequence of 20 qubits
sequence_length = 20

# Generate Alice random bits
alice_bits = generate_random_sequence(sequence_length)

# Generate Alice random bases
alice_bases = generate_random_sequence(sequence_length)

# Encodes Alice qubits
message_qubits = encode_message(alice_bits, alice_bases)

print("Alice Raw Bits:", alice_bits)
print("Alice Bases:", alice_bases)

Alice Raw Bits: [0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 0]
Alice Bases: [1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1]


In [29]:
# ==========================================
# EVE Attacker Intercepts
# ==========================================
def measure_message(message_qubits, bases):
    results = []
    simulator = BasicSimulator()

    for qc, basis in zip(message_qubits, bases):
        if basis == 1:
            qc.h(0) # applies H gate to rotate back to diagonal

        qc.measure(0, 0)
        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)
        measured_bit = int(list(counts.keys())[0])
        results.append(measured_bit)

    return results

# Generate Eve random bases to guess Alice encoding
eve_bases = generate_random_sequence(sequence_length)

# Intercepts and measures Alice message_qubits
eve_measured_bits = measure_message(message_qubits, eve_bases)

# Encode fake qubits for Bob with Eve measured bits and bases
fake_message_qubits = encode_message(eve_measured_bits, eve_bases)

print("Eve Guessed Bases:", eve_bases)
print("Eve Measured Bits:", eve_measured_bits)

Eve Guessed Bases: [1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0]
Eve Measured Bits: [0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1]


In [30]:
# ==========================================
# BOB
# ==========================================
# Generate Bob random bases
bob_bases = generate_random_sequence(sequence_length)

# Bob measure Eve fake qubits instead of Alice qubits
bob_bits = measure_message(fake_message_qubits, bob_bases)

print("Bob Bases:", bob_bases)
print("Bob Measured Bits:", bob_bits)

Bob Bases: [0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0]
Bob Measured Bits: [0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1]


In [31]:
# ==========================================
# ALICE & BOB Communication
# ==========================================
def discard_wrong_value(alice_bases, bob_bases, bits):
    sifted_key = []
    for i in range(len(alice_bases)):
        if alice_bases[i] == bob_bases[i]:
            sifted_key.append(bits[i])
    return sifted_key

alice_key = discard_wrong_value(alice_bases, bob_bases, alice_bits)
bob_key = discard_wrong_value(alice_bases, bob_bases, bob_bits)

print("Alice Key:", alice_key)
print("Bob Key:", bob_key)

# Check for attacker
# Sacrifice the first half of the key.
sacrifice_size = len(alice_key) // 2

alice_size = alice_key[:sacrifice_size]
bob_size = bob_key[:sacrifice_size]

# Calculate how many bits differ
errors = 0
for a_bit, b_bit in zip(alice_size, bob_size):
    if a_bit != b_bit:
        errors += 1

# Calculate the error rate
error_rate = errors / sacrifice_size if sacrifice_size > 0 else 0
print(f"Bits sacrificed: {sacrifice_size} | Errors detected: {errors} | Error Rate: {error_rate}")

THRESHOLD = 0.10
if error_rate > THRESHOLD:
    print("Attacker detected")
else:
  print("Failed to detect attacker")

Alice Key: [1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0]
Bob Key: [0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0]
Bits sacrificed: 7 | Errors detected: 2 | Error Rate: 0.2857142857142857
Attacker detected
